# Notebook 01: Procesos, Hilos y el GIL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/16_computo/code/01_procesos_hilos.ipynb)

**Módulo 16 — Clase 1**

Este notebook acompaña los archivos `01_procesos_y_hilos.md` y `02_secuencial.md`.

Aquí vas a comprobar tres ideas base:

1. **Proceso** = programa en ejecución con memoria propia.
2. **Hilo** = camino de ejecución dentro de un proceso; comparte memoria con otros hilos.
3. **GIL** = en CPython, solo un hilo ejecuta bytecode Python a la vez.

---


In [ ]:
import os
import sys
import time
import threading
import multiprocessing

print(f'Python {sys.version}')
print(f'CPU cores: {os.cpu_count()}')

Python 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
CPU cores: 2


## Los escenarios del chatbot

A lo largo del módulo trabajaremos con dos escenarios de nuestro chatbot:

- **Escenario A** — LLM como API remota: `recv(1ms exec) → BD(50ms wait) → LLM API(1500ms wait) → send(5ms wait)` → I/O-bound → M4
- **Escenario B** — LLM local: `recv(1ms exec) → BD(50ms wait) → inferencia(2000ms exec) → send(5ms wait)` → CPU-bound → M5b

Este notebook establece los conceptos fundamentales que hacen posible esa distinción: procesos, hilos y el GIL.


## Sección 1: Inspeccionando un proceso

Cada programa Python que ejecutas ES un proceso con su propio PID y espacio de memoria.

In [ ]:
print(f'Mi PID (Process ID): {os.getpid()}')
print(f'PID de mi proceso padre: {os.getppid()}')

# Intentar instalar psutil si no está disponible
try:
    import psutil
    proc = psutil.Process(os.getpid())
    mem = proc.memory_info()
    print(f'Memoria RSS: {mem.rss / 1024**2:.1f} MB')
    print(f'Memoria VMS: {mem.vms / 1024**2:.1f} MB')
except ImportError:
    print('psutil no instalado — pip install psutil para más info')

Mi PID (Process ID): 234
PID de mi proceso padre: 90
Memoria RSS: 100.9 MB
Memoria VMS: 648.7 MB


## Sección 2: Inspeccionando hilos

El hilo principal es θ_main. Podemos crear hilos adicionales — todos comparten el heap del proceso.

In [ ]:
print(f'Hilo actual: {threading.current_thread().name}')
print(f'ID del hilo: {threading.current_thread().ident}')
print(f'Hilos activos: {threading.active_count()}')

# Crear un hilo que reporte su identidad
def reportar_identidad(nombre):
    print(f'  [{nombre}] PID={os.getpid()}, Hilo={threading.current_thread().name}, ID={threading.current_thread().ident}')

print('\nLanzando 3 hilos:')
hilos = []
for i in range(3):
    h = threading.Thread(target=reportar_identidad, args=(f'hilo-{i}',))
    hilos.append(h)
    h.start()

for h in hilos:
    h.join()

print('\nObserva: todos los hilos tienen el MISMO PID — son del mismo proceso.')

Hilo actual: MainThread
ID del hilo: 135618310447104
Hilos activos: 5

Lanzando 3 hilos:
  [hilo-0] PID=234, Hilo=Thread-3 (reportar_identidad), ID=135617732142656
  [hilo-1] PID=234, Hilo=Thread-4 (reportar_identidad), ID=135617732142656
  [hilo-2] PID=234, Hilo=Thread-5 (reportar_identidad), ID=135617723749952

Observa: todos los hilos tienen el MISMO PID — son del mismo proceso.


## Sección 3: Demostración del GIL — CPU-bound con threading

**Hipótesis ingenua:** con 2 hilos y 2 cores, una tarea CPU-bound debería tardar la mitad.  
**Resultado real en CPython:** el GIL serializa la ejecución del bytecode Python → casi no hay speedup (e incluso puede empeorar un poco por overhead).

La idea importante es esta:

- **CPU-bound**: el tiempo está en `exec(τ)`  
- **I/O-bound**: el tiempo está en `wait(τ)`  

En una tarea CPU-bound pura, `threading` no rompe el cuello de botella del GIL.


In [ ]:
def tarea_cpu_bound(n, nombre=''):
    """Tarea CPU-bound pura: suma de 0 a n. wait(τ) = ∅"""
    return sum(range(n))

N = 30_000_000  # ajusta si en tu máquina corre demasiado lento o demasiado rápido

# --- Secuencial ---
t0 = time.perf_counter()
r1 = tarea_cpu_bound(N)
r2 = tarea_cpu_bound(N)
t_secuencial = time.perf_counter() - t0

# --- Threading (2 hilos, misma carga total) ---
resultados = {}

def wrapper(nombre, n):
    resultados[nombre] = tarea_cpu_bound(n, nombre)

t0 = time.perf_counter()
h1 = threading.Thread(target=wrapper, args=('h1', N))
h2 = threading.Thread(target=wrapper, args=('h2', N))
h1.start(); h2.start()
h1.join(); h2.join()
t_threading = time.perf_counter() - t0

assert r1 == resultados['h1'] and r2 == resultados['h2']

print(f'Secuencial:  {t_secuencial:.2f}s')
print(f'Threading:   {t_threading:.2f}s')
print(f'Speedup:     {t_secuencial/t_threading:.2f}x  (esperado ingenuamente ~2x, real ~1x por el GIL)')
print()
print('Conclusión: para CPU-bound, threading en Python NO ayuda.')
print('El GIL hace que solo un hilo ejecute bytecode Python a la vez.')


Secuencial:  1.16s
Threading:   1.16s
Speedup:     1.00x  (esperado ~2x, real ~1x por el GIL)

Conclusión: para CPU-bound, threading en Python NO ayuda.
El GIL garantiza que solo un hilo ejecuta bytecode a la vez.


### Extra: CPU-bound con **procesos**

Si queremos paralelismo real para trabajo **CPU-bound**, usamos **procesos** en lugar de hilos.  
Cada proceso tiene su propio intérprete y su propio GIL, así que sí puede correr en paralelo en varios cores.

> Nota: este experimento funciona mejor en Linux/Colab/Jupyter con soporte normal de `multiprocessing`.


In [ ]:
def tarea_cpu_bound_proceso(n, q):
    q.put(sum(range(n)))

# En Linux/Colab normalmente se puede usar fork dentro del notebook.
# En otros entornos puede variar.
if os.name != 'nt':
    ctx = multiprocessing.get_context('fork')
    q = ctx.Queue()

    t0 = time.perf_counter()
    p1 = ctx.Process(target=tarea_cpu_bound_proceso, args=(N, q))
    p2 = ctx.Process(target=tarea_cpu_bound_proceso, args=(N, q))
    p1.start(); p2.start()

    resultados_mp = [q.get(), q.get()]
    p1.join(); p2.join()
    t_multiprocessing = time.perf_counter() - t0

    assert sorted(resultados_mp) == sorted([r1, r2])

    print(f'Multiprocessing: {t_multiprocessing:.2f}s')
    print(f'Speedup vs secuencial: {t_secuencial/t_multiprocessing:.2f}x')
    print('Conclusión: para CPU-bound, procesos sí pueden dar speedup real.')
else:
    print('Demo omitida en este entorno Windows/notebook para evitar problemas con multiprocessing.')


---

## Sección 4: GIL liberado — I/O-bound con threading

Durante `wait(τᵢ)` (I/O), el GIL se libera. Aquí sí hay speedup con threading.

In [ ]:
def tarea_io_bound(duracion, nombre=''):
    """Simula I/O-bound: espera bloqueante. wait(τ) ≠ ∅"""
    time.sleep(duracion)  # durante sleep, el GIL se libera

DURACION = 1.0  # cada tarea "espera" 1s de I/O

# --- Secuencial ---
t0 = time.perf_counter()
tarea_io_bound(DURACION)
tarea_io_bound(DURACION)
t_secuencial = time.perf_counter() - t0

# --- Threading ---
t0 = time.perf_counter()
h1 = threading.Thread(target=tarea_io_bound, args=(DURACION,))
h2 = threading.Thread(target=tarea_io_bound, args=(DURACION,))
h1.start(); h2.start()
h1.join(); h2.join()
t_threading = time.perf_counter() - t0

print(f'Secuencial:  {t_secuencial:.2f}s  (suma: 2 × {DURACION}s)')
print(f'Threading:   {t_threading:.2f}s  (paralelo en I/O, GIL liberado)')
print(f'Speedup:     {t_secuencial/t_threading:.2f}x  (esperado ~2x)')
print()
print('Conclusión: para I/O-bound, threading SÍ ayuda porque el GIL se libera durante sleep/I/O.')

Secuencial:  2.00s  (suma: 2 × 1.0s)
Threading:   1.00s  (paralelo en I/O, GIL liberado)
Speedup:     2.00x  (esperado ~2x)

Conclusión: para I/O-bound, threading SÍ ayuda porque el GIL se libera durante sleep/I/O.


## Sección 5: Aislamiento de memoria — proceso vs hilo

Los **hilos** comparten la memoria del proceso: si un hilo modifica un objeto mutable, los demás pueden ver ese cambio.  
Los **procesos** no comparten memoria por defecto: cada proceso hijo trabaja con su propia copia.

En esta sección vamos a mostrarlo de forma **determinista**, sin depender de una race condition.


In [ ]:
# --- Hilos comparten memoria ---
compartido = {'contador': 0, 'historial': []}

def modificar_desde_hilo(nombre, n):
    for _ in range(n):
        compartido['contador'] += 1
    compartido['historial'].append(f'{nombre} terminó')

hilos = [threading.Thread(target=modificar_desde_hilo, args=(f'hilo-{i}', 1000)) for i in range(2)]
for h in hilos:
    h.start()
for h in hilos:
    h.join()

print('Tras usar hilos sobre el MISMO objeto compartido:')
print(f"contador = {compartido['contador']}")
print(f"historial = {compartido['historial']}")
print('→ Ambos hilos modificaron la misma estructura en memoria.')

print()

# --- Procesos NO comparten memoria ---
def modificar_desde_proceso(objeto, q):
    objeto['contador'] += 1000
    objeto['historial'].append('proceso-hijo terminó')
    q.put(objeto)  # regresamos la copia modificada del hijo

if os.name != 'nt':
    ctx = multiprocessing.get_context('fork')
    dato_padre = {'contador': 0, 'historial': []}
    q = ctx.Queue()
    p = ctx.Process(target=modificar_desde_proceso, args=(dato_padre, q))
    p.start()
    copia_hijo = q.get()
    p.join()

    print('Comparando proceso padre vs proceso hijo:')
    print(f'Padre ({os.getpid()}): {dato_padre}')
    print(f'Hijo  (copia devuelta por queue): {copia_hijo}')
    print('→ El padre conserva su objeto original; el hijo trabajó sobre otra copia.')
else:
    print('Demo de procesos omitida en este entorno Windows/notebook para evitar problemas con multiprocessing.')


Contador tras 2 hilos × 1000 incrementos: 2000
(Nota: puede ser < 2000 debido a race condition — lo veremos en Clase 2)

  Proceso hijo (1877): su copia = 1000
Proceso padre (234): su copia = 0
→ El padre no ve el cambio del hijo: Mem(padre) ∩ Mem(hijo) = ∅


---

## Resumen

| Concepto | Clave |
|---|---|
| Proceso | Tiene su propia memoria aislada |
| Hilo | Comparte memoria del proceso |
| GIL | Solo 1 hilo ejecuta bytecode Python a la vez |
| CPU-bound + threading | Sin speedup relevante |
| CPU-bound + procesos | Sí puede haber paralelismo real |
| I/O-bound + threading | Sí hay speedup porque el GIL se libera durante I/O |
